# 04 - Modelling
**CRISP-DM phase:** Modelling

Four models answer the research questions from notebook 01:

| Model | Question | Data | Type |
|---|---|---|---|
| **S1** | Will this crash involve injury? | CRSS crash table | binary classification |
| **S2** | Will this crash involve serious or fatal injury? | CRSS crash table | binary classification, imbalanced (about 13%) |
| **U1** | Which component group(s) is this complaint about? | complaint narratives | multi-label classification (19 groups) |
| **U2** | Does this complaint describe a crash, fire, injury or death? | complaint narratives | binary classification, imbalanced (about 6-10%) |

For each, several model families are trained and compared against a no-skill baseline. **All tuning and model selection use the validation period only**; test results appear here for completeness and are analysed in notebook 05.

Training is implemented in `src/train_crss.py` (S1, S2) and `src/train_text.py` (U1, U2). The cells below show the exact configuration used (read directly from those scripts) and the results they produced.

## 4.0 Setup
`RUN_TRAINING = False` loads saved results. Retraining everything takes about 3 hours on a 12-core laptop with 15 GB RAM.

In [1]:
from pathlib import Path
import inspect, json, subprocess, sys

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from project_paths import EVALUATION_DIR, TABLE_DIR
import train_crss, train_text, train_crash_deploy

RUN_TRAINING = False
if RUN_TRAINING:
    for script in ["train_crss.py", "train_text.py", "train_crash_deploy.py"]:
        subprocess.run([sys.executable, str(PROJECT_ROOT / "src" / script)], check=True, cwd=PROJECT_ROOT)

pd.set_option("display.max_columns", 40, "display.width", 220, "display.precision", 4)
show_code = lambda obj: display(Markdown("```python\n" + inspect.getsource(obj) + "\n```"))

# Part A - Structured crash models (S1, S2)

## 4.1 Preparing the crash table for the models
Two steps convert the crash table into model input, both fitted on the 2020-2022 training years only:
1. **Remove redundant columns:** constant columns and exact duplicates (e.g. pedestrian counts recorded identically in several tables).
2. **Encode and fill:** the 17 crash-level coded fields are one-hot encoded; numeric and count fields have missing values filled with the training median.

Logistic Regression additionally standardises each column (inside its own pipeline); tree models do not need scaling.

In [2]:
show_code(train_crss.drop_redundant)
show_code(train_crss.preprocessor)

```python
def drop_redundant(X: pd.DataFrame) -> list[str]:
    """Remove constant and exactly duplicated columns (decided on training data only).

    Columns are compared by a fingerprint of their values, which avoids transposing the
    whole training matrix (the transpose needed several GB once the VIN-decode fields were added).
    """
    keep, seen = [], set()
    for c in X.columns:
        v = X[c].to_numpy()
        if pd.Series(v).nunique(dropna=False) <= 1:
            continue
        key = hash(np.nan_to_num(v.astype(np.float64), nan=-1e300).tobytes())
        if key not in seen:
            seen.add(key)
            keep.append(c)
    return keep

```

```python
def preprocessor(cat: list[str], num: list[str]) -> ColumnTransformer:
    """Fitted once per feature set on training years; scaling happens inside the LR pipeline."""
    return ColumnTransformer([
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value=-9)),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32))]), cat),
        ("num", MedianImputer32(), num),
    ], verbose_feature_names_out=False)

```

## 4.2 Candidate models and tuning grid
Model families were chosen to move from simple and interpretable to flexible:

| Family | Why it is included |
|---|---|
| Dummy (prior) | No-skill baseline: always predicts the training rate |
| Logistic Regression | Linear, interpretable baseline; tuned on regularisation strength `C` |
| Decision Tree | Readable rules; tuned on depth |
| Random Forest | Many de-correlated trees; tuned on minimum leaf size |
| Gradient Boosting (histogram) | Sequential trees correcting earlier errors; tuned on leaf count, with early stopping |

Each family is tried with a small grid; the setting with the best **validation ROC-AUC** is kept for that family.

In [3]:
show_code(train_crss.candidates)

```python
def candidates(balanced: bool) -> dict:
    cw = "balanced" if balanced else None
    lr = lambda c: Pipeline([("scale", StandardScaler()), ("lr", LogisticRegression(C=c, max_iter=3000, class_weight=cw))])
    return {
        "Dummy (prior)": [DummyClassifier(strategy="prior")],
        "Logistic Regression": [lr(c) for c in (0.01, 0.1, 1.0)],
        "Decision Tree": [DecisionTreeClassifier(max_depth=d, min_samples_leaf=50, class_weight=cw, random_state=RNG)
                          for d in (6, 10, 14)],
        "Random Forest": [RandomForestClassifier(n_estimators=300, min_samples_leaf=l, max_features="sqrt",
                                                 class_weight=cw, n_jobs=-1, random_state=RNG) for l in (5, 20)],
        "Gradient Boosting": [HistGradientBoostingClassifier(learning_rate=0.08, max_leaf_nodes=n, max_iter=600,
                                                             l2_regularization=1.0, early_stopping=True,
                                                             validation_fraction=0.1, n_iter_no_change=30,
                                                             class_weight=cw, random_state=RNG) for n in (31, 63)],
    }

```

## 4.3 Two feature sets and class weighting
* **Feature sets.** Every model is trained twice: with **all** fields, and with **no post-crash fields** (damage extent, towing, alcohol-test results). Fields recorded after the crash may make the model look better than it would be at an earlier decision point, so the second set measures how much the models depend on them.
* **Class weighting (S2 only).** Serious or fatal crashes are about 13% of cases. Each S2 model is also trained with `class_weight="balanced"`, which weights the rare class more heavily, to test whether it improves detection.
* **Threshold.** For each model the decision threshold is the one that maximises F1 on validation data, not a default 0.5.

> **Initial plan:** S2 was to predict whether a **vehicle defect** contributed to the crash, linking directly to the complaint data.
> **What we found:** a police-recorded vehicle defect appears for only about 1.6% of vehicles (roughly 1,300-1,700 a year), too few to train a reliable model, and the defect field is recorded after investigation.
> **Decision:** S2 predicts **serious or fatal injury** (about 35,000 positive crashes). Defects remain part of the analysis: notebook 02's tests show crashes with a recorded defect are about 12% more likely to be serious (15.0% vs 13.4%, p = 0.0002).

## 4.4 Results: S1 injury crash
One row per model family and feature set, at its best setting. Selection rule: highest **validation PR-AUC** among non-baseline models.

In [4]:
cols = ["feature_set", "class_weight", "model", "params", "n_input_fields", "fit_seconds",
        "train_roc_auc", "validation_roc_auc", "validation_pr_auc", "test_roc_auc", "test_pr_auc", "test_f1", "selected"]
s1 = pd.read_csv(EVALUATION_DIR / "crss_y_injury_model_comparison.csv")
s1[cols]

,feature_set,class_weight,model,params,n_input_fields,fit_seconds,train_roc_auc,validation_roc_auc,validation_pr_auc,test_roc_auc,test_pr_auc,test_f1,selected
0,all,none,Dummy (prior),{},1317,0.0,0.5000,0.5000,0.5148,0.5000,0.5130,NaN,False
1,all,none,Logistic Regression,"{""C"": 0.01}",1317,24.8,0.8915,0.8954,0.9094,0.8860,0.8994,0.8137,False
2,all,none,Decision Tree,"{""max_depth"": 14, ""max_leaf_nodes"": null, ""min...",1317,126.1,0.8926,0.8831,0.8910,0.8837,0.8904,0.8097,False
3,all,none,Random Forest,"{""max_depth"": null, ""max_leaf_nodes"": null, ""m...",1317,168.4,0.9714,0.8980,0.9117,0.8984,0.9112,0.8206,False
4,all,none,Gradient Boosting,"{""max_depth"": null, ""max_leaf_nodes"": 63, ""min...",1317,218.3,0.9278,0.9063,0.9190,0.9063,0.9185,0.8266,True
5,no_post_crash,none,Dummy (prior),{},1264,0.0,0.5000,0.5000,0.5148,0.5000,0.5130,NaN,False
6,no_post_crash,none,Logistic Regression,"{""C"": 0.01}",1264,26.0,0.8811,0.8859,0.9021,0.8764,0.8921,0.8030,False
7,no_post_crash,none,Decision Tree,"{""max_depth"": 14, ""max_leaf_nodes"": null, ""min...",1264,58.8,0.8803,0.8772,0.8885,0.8768,0.8858,0.8024,False
8,no_post_crash,none,Random Forest,"{""max_depth"": null, ""max_leaf_nodes"": null, ""m...",1264,122.3,0.9695,0.8881,0.9045,0.8891,0.9043,0.8121,False
9,no_post_crash,none,Gradient Boosting,"{""max_depth"": null, ""max_leaf_nodes"": 31, ""min...",1264,112.2,0.9031,0.8981,0.9128,0.8979,0.9123,0.8179,False


**Interpretation.** Every model clearly beats the baseline. Gradient Boosting has the best validation scores and is selected. Removing post-crash fields lowers performance only slightly, showing the model relies mainly on how the crash happened.

## 4.5 Results: S2 serious or fatal crash

In [5]:
s2 = pd.read_csv(EVALUATION_DIR / "crss_y_serious_model_comparison.csv")
s2[cols]

,feature_set,class_weight,model,params,n_input_fields,fit_seconds,train_roc_auc,validation_roc_auc,validation_pr_auc,test_roc_auc,test_pr_auc,test_f1,selected
0,all,none,Dummy (prior),{},1317,0.0,0.5000,0.5000,0.1350,0.5000,0.1300,NaN,False
1,all,none,Logistic Regression,"{""C"": 0.01}",1317,21.1,0.8925,0.8668,0.5568,0.8662,0.5438,0.5189,False
2,all,none,Decision Tree,"{""max_depth"": 10, ""max_leaf_nodes"": null, ""min...",1317,54.2,0.8730,0.8598,0.5435,0.8612,0.5328,0.5106,False
3,all,none,Random Forest,"{""max_depth"": null, ""max_leaf_nodes"": null, ""m...",1317,135.4,0.9830,0.8895,0.6100,0.8899,0.5998,0.5477,False
4,all,none,Gradient Boosting,"{""max_depth"": null, ""max_leaf_nodes"": 31, ""min...",1317,128.0,0.9285,0.9000,0.6471,0.9017,0.6421,0.5749,True
5,all,balanced,Logistic Regression,"{""C"": 0.01}",1317,24.1,0.8943,0.8656,0.5417,0.8641,0.5258,0.5140,False
6,all,balanced,Decision Tree,"{""max_depth"": 10, ""max_leaf_nodes"": null, ""min...",1317,55.9,0.8714,0.8539,0.5163,0.8581,0.5082,0.4899,False
7,all,balanced,Random Forest,"{""max_depth"": null, ""max_leaf_nodes"": null, ""m...",1317,116.5,0.9896,0.8885,0.5944,0.8890,0.5851,0.5472,False
8,all,balanced,Gradient Boosting,"{""max_depth"": null, ""max_leaf_nodes"": 31, ""min...",1317,99.3,0.9219,0.8992,0.6407,0.9011,0.6343,0.5759,False
9,no_post_crash,none,Dummy (prior),{},1264,0.0,0.5000,0.5000,0.1350,0.5000,0.1300,NaN,False


**Interpretation.** PR-AUC is the more informative measure here because only about 13% of crashes are serious; the baseline PR-AUC equals that rate. Gradient Boosting is again selected. Class weighting does not improve validation PR-AUC, so the unweighted model is kept: weighting shifts the threshold rather than adding information.

## 4.6 Effect of adding the VIN-decode specifications
The models were first trained without the two VIN-decode tables, then retrained with them (notebook 03, step 3.2). Comparing the selected models shows what the manufacturer specifications (fitted safety systems, weight, power) add.

In [6]:
rows = []
for t, label in [("y_injury", "S1 injury"), ("y_serious", "S2 serious/fatal")]:
    before = pd.read_csv(EVALUATION_DIR / "before_vpic" / f"crss_{t}_model_comparison.csv")
    after = pd.read_csv(EVALUATION_DIR / f"crss_{t}_model_comparison.csv")
    b, a = before[before.selected].iloc[0], after[after.selected].iloc[0]
    rows.append({"target": label, "fields before": b.n_input_fields, "fields after": a.n_input_fields,
                 "validation PR-AUC before": b.validation_pr_auc, "validation PR-AUC after": a.validation_pr_auc,
                 "test ROC-AUC before": b.test_roc_auc, "test ROC-AUC after": a.test_roc_auc})
pd.DataFrame(rows)

,target,fields before,fields after,validation PR-AUC before,validation PR-AUC after,test ROC-AUC before,test ROC-AUC after
0,S1 injury,991,1317,0.9196,0.9190,0.9065,0.9063
1,S2 serious/fatal,991,1317,0.6512,0.6471,0.9018,0.9017


**Interpretation.** The VIN-decode specifications add **no measurable predictive power**: test ROC-AUC and PR-AUC are unchanged to the third decimal for both models, even though about 330 more fields reach the model. The likely reason is that what they describe, such as fitted safety systems, weight and engine size, is already largely captured by fields the models had: **model year** (newer vehicles carry more safety equipment) and **body type**. Logistic Regression actually loses a little accuracy, because extra weak fields add noise to a linear model. The specifications are kept in the final models so that no documented field is excluded by hand, and this test shows the result does not depend on that choice.

# Part B - Text models (U1, U2)

## 4.7 Turning narratives into features: TF-IDF
Each narrative becomes a vector of **TF-IDF** weights over words and two-word phrases (e.g. "air bags", "lost power"):
* **TF** (term frequency): how often a term appears in this complaint (log-scaled, so repetition counts less).
* **IDF** (inverse document frequency): terms that appear in many complaints, such as "vehicle", get less weight.

Settings: unigrams and bigrams, terms in at least 5 complaints and at most half of them, 300,000 terms at most. The vocabulary is **fitted on the 741,623 training complaints only**.

In [7]:
src = inspect.getsource(train_text.main)
print(src[src.index("vec = TfidfVectorizer"):src.index("Xtr = vec.fit_transform")])

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.5, max_features=300000, sublinear_tf=True,
                          lowercase=True, strip_accents="unicode", token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9]+\b",
                          dtype=np.float32)
    


## 4.8 Candidate text models
| Family | Why it is included |
|---|---|
| Dummy (prior) | No-skill baseline |
| Complement Naive Bayes | Fast probabilistic model suited to word counts and imbalanced classes |
| Logistic Regression | Linear model; coefficients show which terms push a prediction |
| Linear SVM | Linear maximum-margin classifier; strong standard for high-dimensional text |

U1 is multi-label, so each family is wrapped in **one-vs-rest**: one binary model per component group, each with its own validated threshold. If no group passes its threshold, the highest-scoring group is still recommended, so every complaint receives a routing suggestion.

In [8]:
show_code(train_text.candidates)

```python
def candidates(task: str) -> dict:
    wrap = (lambda m: OneVsRestClassifier(m, n_jobs=4)) if task == "U1" else (lambda m: m)
    return {
        "Dummy (prior)": [wrap(DummyClassifier(strategy="prior"))],
        "Complement Naive Bayes": [wrap(ComplementNB(alpha=a)) for a in (0.1, 0.5)],
        "Logistic Regression": [wrap(LogisticRegression(C=c, solver="liblinear", max_iter=1000)) for c in (1.0, 4.0)],
        "Linear SVM": [wrap(LinearSVC(C=c)) for c in (0.1, 0.5)],
    }

```

## 4.9 Results: U2 serious-incident flag

In [9]:
u2 = pd.read_csv(EVALUATION_DIR / "text_U2_model_comparison.csv")
u2.drop(columns=["params"])

,task,model,fit_seconds,validation_pr_auc,train_roc_auc,train_pr_auc,train_precision,train_recall,train_f1,validation_roc_auc,validation_precision,validation_recall,validation_f1,test_roc_auc,test_pr_auc,test_precision,test_recall,test_f1,overfit_gap_train_minus_val,selected
0,U2,Dummy (prior),0.0,0.0639,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,U2,Complement Naive Bayes,1.1,0.8000,0.9543,0.8527,0.8814,0.7199,0.7925,0.9392,0.8286,0.6913,0.7537,0.9453,0.7976,0.8099,0.6962,0.7487,0.0527,False
2,U2,Logistic Regression,52.2,0.8643,0.9947,0.9659,0.9656,0.8214,0.8877,0.9657,0.8759,0.7489,0.8075,0.9661,0.8550,0.8529,0.7469,0.7964,0.1016,False
3,U2,Linear SVM,31.8,0.8670,0.9895,0.9434,0.9368,0.8150,0.8717,0.9647,0.8579,0.7697,0.8114,0.9650,0.8580,0.8342,0.7699,0.8008,0.0764,True


**Interpretation.** The baseline PR-AUC is about 0.06 (the validation positive rate). Linear SVM has the best validation PR-AUC and is selected; Logistic Regression is very close. Naive Bayes trails both but is far above the baseline.

## 4.10 Results: U1 component routing

In [10]:
u1 = pd.read_csv(EVALUATION_DIR / "text_U1_model_comparison.csv")
u1[["model", "fit_seconds", "train_macro_f1", "validation_micro_f1", "validation_macro_f1",
    "validation_top1_accuracy", "test_micro_f1", "test_macro_f1", "test_top1_accuracy", "selected"]]

,model,fit_seconds,train_macro_f1,validation_micro_f1,validation_macro_f1,validation_top1_accuracy,test_micro_f1,test_macro_f1,test_top1_accuracy,selected
0,Dummy (prior),4.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
1,Complement Naive Bayes,16.9,0.6949,0.6608,0.6175,0.7253,0.6436,0.5792,0.7196,False
2,Logistic Regression,583.6,0.8314,0.7304,0.7076,0.8259,0.7256,0.6763,0.8330,False
3,Linear SVM,390.6,0.7864,0.7443,0.7205,0.8360,0.7396,0.6911,0.8441,True


**Interpretation.** *Top-1 accuracy* is the share of complaints whose highest-ranked group is one of their true groups; *macro-F1* averages all 19 groups equally, so small groups count as much as large ones. Linear SVM is selected on validation macro-F1. Logistic Regression fits the training data more closely but generalises slightly worse (a larger train-validation gap; see notebook 05).

# Part C - A compact crash model for the application

> **Initial plan:** deploy the selected S1/S2 research models in the web application.
> **What we found:** those models use about 1,300 fields aggregated from 28 tables; nobody can enter that by hand for a new crash.
> **Decision:** train compact versions on **30 inputs an analyst would have from a police report** (crash type, light, weather, speeds, vehicle and occupant counts, rollover, restraint use, and so on), using the same split and selection rule, and measure the accuracy given up for usability.

In [11]:
show_code(train_crash_deploy.build)
pd.read_csv(EVALUATION_DIR / "crash_deploy_comparison.csv")

```python
def build(df: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=df.index)
    for f, (col, *_rest) in CATEGORICAL.items():
        X[f] = df[col].where(df[col] >= 0)  # -1 = code outside training vocabulary -> unknown
    for f, (fn, *_rest) in NUMERIC.items():
        X[f] = fn(df).astype("float32")
    return X

```

,target,model,inputs,train_roc_auc,validation_roc_auc,validation_pr_auc,test_roc_auc,test_pr_auc,test_f1,threshold,full_model_test_roc_auc,full_model
0,y_injury,Compact Logistic Regression,30,0.8224,0.8357,0.8614,0.8359,0.8604,0.7590,0.4283,0.9063,Gradient Boosting (1317 fields)
1,y_injury,Compact Gradient Boosting,30,0.8548,0.8496,0.8744,0.8494,0.8732,0.7704,0.4353,0.9063,Gradient Boosting (1317 fields)
2,y_serious,Compact Logistic Regression,30,0.8310,0.8324,0.4882,0.8305,0.4736,0.4768,0.2220,0.9017,Gradient Boosting (1317 fields)
3,y_serious,Compact Gradient Boosting,30,0.8746,0.8539,0.5404,0.8526,0.5279,0.5052,0.2320,0.9017,Gradient Boosting (1317 fields)


**Interpretation.** The compact Gradient Boosting models are selected for the application. They give up some ROC-AUC compared with the full research models, the measured price of a form a person can complete. The research models remain the reference results for RQ1 and RQ2.

**Hand-off.** Notebook 05 evaluates the selected models on the untouched test data, checks for overfitting and explains what drives their predictions.